In [ ]:
import ee
from geemap import Map
from multiprocessing import Pool

try:
    ee.Initialize(project="ee708-rainfall-downscaling")
except:
    ee.Authenticate(auth_mode="localhost")
    ee.Initialize(project="ee708-rainfall-downscaling")

In [ ]:
START_DATE ='1981-01-01'
END_DATE = '2020-07-09'# Exclusive. ERA5 data is available till 2020-07-09

In [ ]:
bands = [
    "total_precipitation_sum",
    "dewpoint_temperature_2m",
    "skin_temperature",
    "snow_albedo",
    "snow_cover",
    "snow_depth",
    "snow_density",
    "snowfall_sum",
    "snowmelt_sum",
    "temperature_of_snow_layer",
    "skin_reservoir_content",
    "runoff_sum",
    "snow_evaporation_sum",
    "sub_surface_runoff_sum",
    "surface_runoff_sum",
    "total_evaporation_sum",
    "surface_pressure",
    "potential_evaporation_sum"
]

In [ ]:
dataset = ee.ImageCollection("ECMWF/ERA5/DAILY").filterDate(START_DATE, END_DATE)
# dataset = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").filterDate(START_DATE, END_DATE)

In [ ]:
dataset.first().getInfo()

In [ ]:
himalayas_coords = [
    [76.47717699953913, 37.82664513320144],
    # [77.39147302732823, 36.47134074092161],
    # [79.67486846517245, 36.42314935342572],
    [80.90026755062496, 35.571803592889346],
    # [79.94743070144554, 32.78366923793975],
    # [81.87950044641099, 30.949760527513547],
    # [88.91295141369213, 28.71385417780965],
    # [94.50093992837051, 30.22256514409202],
    [97.21511065845843, 29.790425258748012],
    [97.9846584261799, 28.219530525561403],
    [97.17357167816135, 27.0017376993619],
    [93.32013533599073, 21.895638006891925],
    [92.12434489366078, 21.640209347541635],
    [91.44246786493959, 22.391938555703494],
    [87.81972158780012, 21.51648763575694],
    [87.17733874211916, 20.269295275574773],
    [80.95032103492291, 15.681782913430276],
    [80.28755447922079, 10.745717535241797],
    [80.03787822576705, 10.131854192880327],
    [77.5784805324031, 7.706238742849392],
    [76.24263442936031, 8.706332281101915],
    [74.43180260094009, 12.350805112400755],
    [72.39906488921469, 18.301660670307868],
    [67.94905459916471, 23.54293033675648],
    # [69.43726122140745, 27.764648638633588],
    [68.21883265960672, 23.247691821952614],
    [ 67.97708437455793, 23.839881647923207],
    [68.4698586761021, 24.46504974206444],
    [69.200023021483, 24.63005695508358],
    [70.58524083020568, 24.717456092256207],
    [69.16683481739948, 26.453152924415633],
    [69.00152321474258, 27.24672911668973],
    [69.91371433766922, 28.321561130479736],
    [71.14725193419356, 28.44732698104629],
    [73.27561368771326, 32.25007221718046],
    [71.84380570344365, 35.84156760093365],
    [73.14597692999519, 37.490491017813504],
    # [86.84464490510447, 35.64673507322577],  # Closing the polygon
]

# Export the image, specifying the CRS, transform, and region.
center_lat = 29.484407
center_lon = 83.211032
# center_point = ee.Geometry.Point([center_lon, center_lat])

# Create a 2500x2500 km square area around the center point.
# The buffer function takes a radius in meters.
# For a square, we create a buffer and then get the bounding box.
# 2500 km / 2 = 1250 km = 1,250,000 meters
# region = center_point.buffer(1250000).bounds()
region = ee.Geometry.Polygon(himalayas_coords)

In [ ]:
map = Map(center=[center_lat, center_lon], zoom=5, basemap="HYBRID")
vis_params = {
    "min": 0.0,
    "max": 0.1,
    "palette":['#ffffff', '#00ffff', '#0080ff', '#da00ff', '#ffa400', '#ff0000'],
    # "palette": "palette"
}
map.addLayer(dataset.select("total_precipitation").first(), vis_params, "ERA5")
# for band in bands:
#     map.addLayer(dataset.select("total_precipitation_sum").first(), vis_params, band)
map

In [ ]:
EXPORT_ERA5_DAILY_AGGR = False
EXPORT_ERA5_DAILY = True

In [ ]:
# Error in 1980-12-29
NO_DATA_VALUE = 0
dataset_list = dataset.toList(dataset.size())
n = ee.Number(dataset.size()).getInfo()

if EXPORT_ERA5_DAILY_AGGR:
    def export_files(band):
        for i in range(n):

            try:
                image = ee.Image(dataset_list.get(i))
                image = image.select(band)
                projection = image.projection().getInfo()
                date = image.get('system:time_start').getInfo()
                date_str = ee.Date(date).format('YYYY-MM-dd').getInfo()
                task = ee.batch.Export.image.toDrive(
                    image=image,
                    description=f'ERA5_daily_{band}_{date_str}',
                    folder=f'ERA5_daily_Aggr/{band}',
                    crs=projection['crs'],
                    crsTransform=projection['transform'],
                    maxPixels=1e13,
                    region=region,
                    formatOptions={'noData': NO_DATA_VALUE},
                )

                task.start()

                print(f'Started export for {band}_{date_str}')
            except Exception as e:
                print(e)

    with Pool() as pool:
        pool.map(export_files, bands)
        
if EXPORT_ERA5_DAILY:
    for i in range(n):
        image = ee.Image(dataset_list.get(i))
        try:
            projection = image.projection().getInfo()
            date = image.get('system:time_start').getInfo()
            date_str = ee.Date(date).format('YYYY-MM-dd').getInfo()

            task = ee.batch.Export.image.toDrive(
                image=image,
                description=f'ERA5_daily_{date_str}',
                folder=f'ERA5_daily_Aggr',
                crs=projection['crs'],
                crsTransform=projection['transform'],
                maxPixels=1e13,
                region=region,
                formatOptions={'noData': NO_DATA_VALUE},
            )

            task.start()

            print(f'Started export for {date_str}')
        except Exception as e:
            print(e)
            break